# 01 — Inspect TIFF channels and prepare the dataset

This notebook is the first executable step of the pipeline. It discovers or loads the manifest, inspects TIFF metadata, verifies the automatic GFAP/DAPI mapping, and runs the same `prepare_dataset.py` command used in production.

Expected result: extracted GFAP and DAPI arrays, nucleus instance labels, binary nucleus masks, proximity maps, QC images, and an updated manifest. Raw TIFF files are never modified.

Run the cells from top to bottom. Edit only the values in the **User settings** cell.

## Environment setup

Start Jupyter from either the repository root or the `notebooks/` directory. This cell locates the repository and makes `src/astroseg` importable without relying on the current working directory.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

candidates = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Open this notebook from the repository root or notebooks directory.')
os.chdir(PROJECT_ROOT)
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print('Project root:', PROJECT_ROOT)
print('Python:', sys.executable)

## User settings

Set `REBUILD_MANIFEST=True` only when creating a new manifest from the TIFF files currently under `data/raw/`. Rebuilding replaces the selected manifest, so do not use it after adding annotation metadata unless that replacement is intentional.

In [ ]:
RAW_DIRECTORY = Path('data/raw')
MANIFEST_PATH = Path('data/metadata/manifest.csv')
INTERIM_DIRECTORY = Path('data/interim')
REBUILD_MANIFEST = False
RUN_PREPARATION = True

# Choose an image after the manifest is loaded. None selects the first row.
IMAGE_ID = None

raw_tiffs = sorted(p for p in RAW_DIRECTORY.rglob('*') if p.suffix.lower() in {'.bmp', '.tif', '.tiff'})
print(f'Found {len(raw_tiffs)} BMP/TIFF file(s) under {RAW_DIRECTORY}')
for path in raw_tiffs[:10]:
    print(' -', path)
if not raw_tiffs:
    raise FileNotFoundError('Place at least one .bmp, .tif, or .tiff file under data/raw/.')

## Build or load the manifest

The manifest has one row per source image. It is the source of truth for image paths, channels, nucleus labels, annotation lifecycle state, and train/validation/test assignment.

In [ ]:
if REBUILD_MANIFEST or not MANIFEST_PATH.is_file():
    command = [
        sys.executable, 'scripts/build_manifest.py',
        '--raw-dir', str(RAW_DIRECTORY),
        '--output', str(MANIFEST_PATH),
    ]
    subprocess.run(command, check=True)

from astroseg.io import get_channel, load_manifest, load_ome_tiff
from astroseg.preprocessing import percentile_normalize, select_model_channels

manifest = load_manifest(MANIFEST_PATH)
print(f'Manifest contains {len(manifest)} image(s).')
manifest

## Inspect one TIFF before batch preparation

For named OME-TIFF data, biological channel names come from metadata. Standard RGB TIFFs receive `Red`, `Green`, and `Blue`; Blue is used as DAPI and the stronger Red/Green signal is selected as GFAP. Explicit manifest values always take priority.

In [ ]:
selected_id = IMAGE_ID or str(manifest.iloc[0]['image_id'])
matches = manifest.loc[manifest['image_id'] == selected_id]
if len(matches) != 1:
    raise ValueError(f'IMAGE_ID must identify one manifest row; found {len(matches)}')
row = matches.iloc[0]

def resolve_manifest_path(value, manifest_path=MANIFEST_PATH):
    path = Path(str(value))
    for candidate in (path, manifest_path.parent / path):
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(value)

source_path = resolve_manifest_path(row['path'])
microscopy = load_ome_tiff(source_path)
selection = select_model_channels(microscopy, str(row['gfap_channel']), str(row['dapi_channel']))
gfap = get_channel(microscopy, selection.gfap_channel)
dapi = get_channel(microscopy, selection.dapi_channel)

print('Image ID:', selected_id)
print('Source:', source_path)
print('Channel-first shape:', microscopy.image.shape)
print('dtype:', microscopy.image.dtype)
print('Channel names:', microscopy.channel_names)
print('Pixel size (um):', microscopy.pixel_size_um)
print('Selected GFAP / DAPI:', selection.gfap_channel, '/', selection.dapi_channel)
print('Selection method:', selection.method)
print('Red / Green signal scores:', selection.red_score, '/', selection.green_score)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

channel_count = microscopy.image.shape[0]
figure, axes = plt.subplots(1, channel_count, figsize=(5 * channel_count, 5), squeeze=False)
for index, axis in enumerate(axes.flat):
    axis.imshow(percentile_normalize(microscopy.image[index]), cmap='gray')
    name = microscopy.channel_names[index] or f'Channel {index}'
    axis.set_title(name)
    axis.axis('off')
figure.suptitle(f'{selected_id}: all available channels')
figure.tight_layout()
plt.show()

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(percentile_normalize(gfap), cmap='gray')
axes[0].set_title(f'GFAP: {selection.gfap_channel}')
axes[1].imshow(percentile_normalize(dapi), cmap='gray')
axes[1].set_title(f'DAPI: {selection.dapi_channel}')
for axis in axes:
    axis.axis('off')
figure.tight_layout()
plt.show()

## Run automatic preparation

This invokes the tested CLI rather than reimplementing it in the notebook. It processes every manifest row, selects channels, detects DAPI nuclei using classical Otsu plus watershed, creates derived model inputs and QC, and updates channel/nucleus paths in the manifest.

In [ ]:
if RUN_PREPARATION:
    command = [
        sys.executable, 'scripts/prepare_dataset.py',
        '--manifest', str(MANIFEST_PATH),
        '--output-dir', str(INTERIM_DIRECTORY),
    ]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Preparation skipped because RUN_PREPARATION=False')

manifest = load_manifest(MANIFEST_PATH)
manifest[['image_id', 'path', 'gfap_channel', 'dapi_channel', 'cellpose_mask_path']]

## Review the preparation report and QC montage

The report records channel selection, nucleus thresholds, instance counts, foreground fraction, and detector parameters. A scientifically usable run requires visual QC when imaging conditions change.

In [ ]:
import pandas as pd

report_path = INTERIM_DIRECTORY / 'qc/preparation_report.csv'
if not report_path.is_file():
    raise FileNotFoundError(f'Preparation report not found: {report_path}')
report = pd.read_csv(report_path)
report

In [ ]:
montage_path = INTERIM_DIRECTORY / 'qc' / f'{selected_id}_montage.png'
if not montage_path.is_file():
    raise FileNotFoundError(montage_path)
montage = plt.imread(montage_path)
plt.figure(figsize=(15, 10))
plt.imshow(montage)
plt.title(f'Preparation QC: {selected_id}')
plt.axis('off')
plt.show()

## Completion checklist

Before continuing to notebook 02, verify:

- GFAP and DAPI were mapped to the intended biological channels.
- The nucleus mask is aligned with DAPI and contains plausible instances.
- `data/interim/qc/preparation_report.csv` contains one row per image.
- The manifest has non-empty `gfap_channel`, `dapi_channel`, and `cellpose_mask_path` values.

Continue with `02_visualize_nucleus_masks.ipynb` for detailed nucleus QC.